# 03 — Fields Model & OCR Setup

Trains the second YOLOv8n model to detect all 15 card fields (front + back
combined), and sets up PaddleOCR for Arabic text recognition.

In [ ]:
# Pin paddlepaddle version — newer releases break paddleocr's API
# (AttributeError: 'AnalysisConfig' object has no attribute 'set_optimization_level')
!pip install paddlepaddle==3.2.2 -q
!pip install paddleocr -q

In [ ]:
from paddleocr import PaddleOCR
ocr = PaddleOCR(use_angle_cls=True, lang='ar')

In [ ]:
from ultralytics import YOLO

model_fields = YOLO('yolov8n.pt')

results_fields = model_fields.train(
    data='/kaggle/working/all-fields-dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    name='fields_model'
)

**Result:** mAP50 = 0.977 overall. mAP50-95 = 0.71 (lower — expected with only ~120 training images across 15 classes; box localization is less precise than for the 2-class crop model).

In [ ]:
# Load best weights
model_fields = YOLO('/kaggle/working/runs/detect/fields_model/weights/best.pt')

In [ ]:
# Per-field padding when cropping each field — some fields (e.g. address)
# need more margin than others to avoid clipping characters
FIELD_PADDING = {
    "address": 15, "name": 12, "job": 10, "religion": 5,
    "martial_state": 10, "husband": 1, "education": 10,
    "national_id": 5, "birth_date": 8, "issue_date": 8,
    "expire_date": 8, "gender": 5
}

def detect_fields(cropped_card_image, model, conf_threshold=0.25, iou_threshold=0.5):
    """
    Detects all fields inside a cropped card image.
    Returns {field_name: {"image": crop, "confidence": float, "bbox": [...]}}.
    Keeps only the highest-confidence box per field name.
    """
    results = model.predict(source=cropped_card_image, conf=conf_threshold, iou=iou_threshold, verbose=False)

    fields_dict = {}
    if len(results[0].boxes) == 0:
        print("No fields detected")
        return fields_dict

    img_h, img_w = cropped_card_image.shape[:2]

    for box in results[0].boxes:
        class_id = int(box.cls[0])
        field_name = model.names[class_id]
        confidence = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()

        padding = FIELD_PADDING.get(field_name, 1)
        x1 = max(0, int(x1) - padding)
        y1 = max(0, int(y1) - padding)
        x2 = min(img_w, int(x2) + padding)
        y2 = min(img_h, int(y2) + padding)

        field_crop = cropped_card_image[y1:y2, x1:x2]

        if field_name not in fields_dict or confidence > fields_dict[field_name]["confidence"]:
            fields_dict[field_name] = {"image": field_crop, "confidence": confidence, "bbox": [x1, y1, x2, y2]}

    return fields_dict

### Quick test — field detection on a cropped card

In [1]:
# cropped_img here comes from crop_card() in notebook 02
fields = detect_fields(cropped_img, model_fields)

for field_name, data in fields.items():
    print(f"{field_name:15s} | confidence: {data['confidence']:.3f}")

image           | conf: 0.977
country         | conf: 0.954
name            | conf: 0.938
address         | conf: 0.868
birth_date      | conf: 0.775
national_id     | conf: 0.666
doc_type        | conf: 0.660
